# Train on a ZINC stream

**Purpose:** Train on a ZINC stream.

**Before you start:** NSPPK, RDKit, and the selected ZINC CSV. Use the Python environment prepared by [setup](../setup.ipynb).

**Results:** Model, streaming statistics, and sample plots.

Run the cells in order, reviewing the configuration before starting the main work. Data stays under `notebooks/datasets`; models and outputs use the project’s artifact folders.


Load the training tools and configure CPU thread limits.


In [ ]:
print('Load the training tools and configure CPU thread limits.')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import random

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import numpy as np
from IPython.core.display import HTML

HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell; text-align: center; vertical-align: middle;}</style>')

from conditional_node_field_graph_generator.notebooks import configure_notebook, download_zinc_dataset
globals().update(configure_notebook(require_nsppk=True, print_torch=True))

from abstractgraph_graphicalizer.chem import draw_molecules
from conditional_node_field_graph_generator.extensions.demo import show_molecules
from conditional_node_field_graph_generator.extensions.demo.pipeline import build_graph_generator


Choose the ZINC file, streaming batch sizes, and training budget.


In [ ]:
print('Choose the ZINC file, streaming batch sizes, and training budget.')
ZINC_DATA_ROOT = NOTEBOOK_DATA_ROOT / 'zinc'
ZINC_SIZE = 'zinc18'
ZINC_FILENAME = f'{ZINC_SIZE}.csv'
STREAM_LIMIT = 0.9
WARMUP_SIZE = 2049
STREAM_BATCH_SIZE = 128
MAXIMUM_EPOCHS = 256
EMBEDDING_DIM = 64
VERBOSE = 2
MODEL_NAME = f'{ZINC_SIZE}-streaming-d{EMBEDDING_DIM}-s{STREAM_LIMIT}-w{WARMUP_SIZE}-b{STREAM_BATCH_SIZE}-e{MAXIMUM_EPOCHS}'
DECODER_N_JOBS = 1
STREAM_SNAPSHOT_EVERY_N_BATCHES = 20
URI = ZINC_DATA_ROOT / ZINC_FILENAME
RANDOM_SEED = 7
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

Build the generator for training directly from the CSV.


In [ ]:
print('Build the generator for training directly from the CSV.')
# Recurrent energy setup. Set node_field_mode to "baseline" for the original model.
NODE_FIELD_CONFIG = {
    "node_field_mode": "recurrent_energy",
    "recurrent_hidden_dimension": None,  # None uses latent_embedding_dimension.
    "recurrent_training_steps": 8,
    "recurrent_detach_interval": 4,  # None keeps full backpropagation through memory.
    "recurrent_update_scale": 1.0,
    "recurrent_initial_state": "zeros",
    "recurrent_state_normalization": True,
    "recurrent_corruption_schedule": "annealed",  # Also "constant" or "none".
    "recurrent_sigma_min": 0.02,
    "recurrent_sigma_max": None,  # None uses node_field_sigma.
    "recurrent_supervise_all_steps": True,
    "recurrent_loss_discount": 1.0,
}

# Keep new model artifacts distinct from existing baseline checkpoints.
MODEL_NAME = (
    MODEL_NAME.removesuffix("-baseline").removesuffix("-recurrent_energy")
    + "-" + NODE_FIELD_CONFIG["node_field_mode"]
)

graph_generator = build_graph_generator(
    **NODE_FIELD_CONFIG,
    latent_embedding_dimension=EMBEDDING_DIM,
    number_of_transformer_layers=2,
    transformer_attention_head_count=4,
    maximum_epochs=MAXIMUM_EPOCHS,
    batch_size=STREAM_BATCH_SIZE,
    verbose=VERBOSE,
    decoder_n_jobs=DECODER_N_JOBS,
    artifact_root=ARTIFACT_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    model_name=MODEL_NAME,
    model_dir=SAVED_GENERATOR_ROOT,
    stream_snapshot_every_n_batches=STREAM_SNAPSHOT_EVERY_N_BATCHES,
)
graph_generator.stream_batch_timeout_seconds = None
graph_generator.graph_decoder.diagnostic_graph_renderer = draw_molecules


Train from the ZINC stream and report how many graphs were accepted. This may take a long time.


In [ ]:
print('Train from the ZINC stream and report how many graphs were accepted. This may take a long time.')


graph_generator.fit_from_stream(
    URI,
    "zinc_csv",
    warmup_size=WARMUP_SIZE,
    batch_size=STREAM_BATCH_SIZE,
    limit=STREAM_LIMIT,
    random_state=RANDOM_SEED,
    verbose=VERBOSE,
)

print('stream_seen_ =', graph_generator.stream_seen_)
print('stream_warmup_count_ =', graph_generator.stream_warmup_count_)
print('stream_training_seen_ =', graph_generator.stream_training_seen_)
print('stream_training_accepted_ =', graph_generator.stream_training_accepted_)
print('stream_training_skipped_ =', graph_generator.stream_training_skipped_)
print('stream_acceptance_rate_ =', graph_generator.stream_acceptance_rate_)


List saved generators available for later reuse.


In [ ]:
print('List saved generators available for later reuse.')
from conditional_node_field_graph_generator.persistence import (
    list_saved_graph_generators,
    load_graph_generator,
)
list_saved_graph_generators(SAVED_GENERATOR_ROOT)

Optionally load a saved generator.


In [ ]:
print('Optionally load a saved generator.')
# Resume later with a copied filename from the save cell. 
if False:
    MODEL_FILENAME = 'zinc-d64-n10000-size10-18.pkl'  # Replace with your chosen filename from the list above.
    graph_generator = load_graph_generator(MODEL_FILENAME, model_dir=SAVED_GENERATOR_ROOT)

Generate molecules without feasibility filtering.


In [ ]:
print('Generate molecules without feasibility filtering.')
raw_samples = graph_generator.sample(
    n_samples=7,
    feasibility_effort=0,
    feasibility_filter='none',
)
show_molecules(raw_samples, n=7, title='Streaming ZINC samples without feasibility filtering')


Generate molecules with strict feasibility filtering.


In [ ]:
print('Generate molecules with strict feasibility filtering.')
if graph_generator.feasibility_estimator is None:
    raise RuntimeError('Feasibility estimator is unavailable in this environment.')

filtered_samples = graph_generator.sample(
    n_samples=7,
    feasibility_effort=2,
    feasibility_filter='strict',
)
show_molecules(filtered_samples, n=7, title='Streaming ZINC samples with feasibility filtering')
